# TP3 -- ANOVA à un facteur

**Statistique Mathématique 3 -- L3 MIASHS**

---

## Introduction

Cette séance étudie le **lien potentiel entre l'abolition de la peine de mort et le taux de criminalité** (données FBI, 2009), puis l'**absorption de matière grasse par des beignets** (ANOVA à 1 facteur).

---
## 1. Acquisition des données (Peine de mort)

In [ ]:
data = read.csv2("murderusa.csv")
head(data)

In [ ]:
txcrime09 = data$tauxmeurtres
PDMabolie09 = data$PDMabolie09

# Sous-groupes
abolie09 = txcrime09[PDMabolie09 == "oui"]
nonabolie09 = txcrime09[PDMabolie09 == "non"]
abolie09 = as.numeric(abolie09)
nonabolie09 = as.numeric(nonabolie09)

cat("États ayant aboli :", length(abolie09))
cat("\nÉtats n'ayant pas aboli :", length(nonabolie09))

## 2. Statistiques descriptives

In [ ]:
# Boîtes à moustaches
boxplot(abolie09, nonabolie09,
        main = "Taux de criminalité selon l'abolition",
        col = c("orange", "red"),
        names = c("Abolie", "Non Abolie"),
        ylab = "Taux de meurtres pour 1000 hab.")

**Observations :** Les indicateurs sont plus bas dans les états ayant aboli. Mais **aucun lien de cause à effet** ne peut être dégagé.

In [ ]:
cat("Moyenne (abolie) :", mean(abolie09))
cat("\nMoyenne (non abolie) :", mean(nonabolie09))

## 3. ANOVA

In [ ]:
summary(aov(txcrime09 ~ PDMabolie09))

In [ ]:
anova(lm(txcrime09 ~ PDMabolie09))

**Conclusion :** p-value trop grande → on **ne peut pas** affirmer un lien significatif.

## 4. Retrouver les résultats à la main

In [ ]:
# SCM
SCM = length(abolie09) * (mean(abolie09) - mean(txcrime09))^2 +
      length(nonabolie09) * (mean(nonabolie09) - mean(txcrime09))^2
cat("SCM =", round(SCM, 6))

# SCE
SCE = sum((abolie09 - mean(abolie09))^2) +
      sum((nonabolie09 - mean(nonabolie09))^2)
cat("\nSCE =", round(SCE, 6))

# F observé
Fobs = SCM / (SCE / 48)
cat("\nF observé =", round(Fobs, 4))

# p-value
cat("\np-value =", round(1 - pf(Fobs, 1, 48), 4))

---
## 5. Et en 2016 ?

In [ ]:
crime16 = read.csv("crime16.csv", sep = ";")
head(crime16)
boxplot(crime16$crime_nonabo, crime16$crime_abo,
        names = c("Non abolie", "Abolie"),
        main = "Taux de criminalité en 2016",
        col = c("red", "orange"))

---
## 6. Beignets et matière grasse

### 6.1 Contexte

4 matières grasses, 6 fournées de 24 beignets. On mesure la quantité de graisse absorbée (valeurs - 100g).

### 6.2 Acquisition des données

In [ ]:
rm(list = ls())
donnees <- read.table("beignets.txt", header = TRUE)
head(donnees)
X = donnees[, "Gras"]
Groupe = factor(donnees[, "Gr"])
cat("Niveaux :", levels(Groupe))

### 6.3 Test de normalité par groupe

In [ ]:
tapply(X, Groupe, shapiro.test)

Pas de rejet de normalité → on peut utiliser l'ANOVA.

### 6.4 Tableau d'ANOVA

In [ ]:
summary(aov(X ~ Groupe))
anova(lm(X ~ Groupe))

**Conclusion :** Rejet de l'égalité des moyennes → effet **significatif** du type de graisse.

### 6.5 Contrastes

Test : graisses animales (1,2) vs végétales (3,4). Contraste : $\lambda = (1, 1, -1, -1)$.

In [ ]:
lambda.i = c(1, 1, -1, -1)
xbar.i = tapply(X, Groupe, mean)
n = length(X)
n.i = tapply(rep(1, n), Groupe, sum)
a = length(levels(Groupe))

Q.i = tapply(X, Groupe, var) * (n.i - 1)
SCE = sum(Q.i)
dl = n - a
CME = SCE / dl

phi = sum(xbar.i * lambda.i)
s.phi = sqrt(CME * sum(lambda.i^2 / n.i))
tvalue = (phi - 0) / s.phi
pvaleur = pt(tvalue, dl)
pvaleur = 2 * min(pvaleur, 1 - pvaleur)

cat("Contraste φ̂ =", round(phi, 2))
cat("\nt =", round(tvalue, 4))
cat("\np-value =", round(pvaleur, 4))

La nature de la matière grasse a une **influence significative**.

### 6.6 Construction du tableau d'ANOVA à la main

In [ ]:
a <- length(levels(Groupe))
n <- length(X)
n.i <- tapply(rep(1, n), Groupe, sum)
xbar.i <- tapply(X, Groupe, mean)
Q.i <- tapply(X, Groupe, var) * (n.i - 1)

SCT <- (n - 1) * var(X)
SCE <- sum(Q.i)
SCM <- SCT - SCE
dlE <- n - a
dlM <- a - 1
CMM <- SCM / dlM
CME <- SCE / dlE
F.obs <- CMM / CME
seuil.crit <- 1 - pf(F.obs, dlM, dlE)

Anova.tableau <- matrix(
  c(SCM, SCE, SCT, dlM, dlE, NA, CMM, CME, NA, F.obs, NA, NA, seuil.crit, NA, NA),
  nrow = 3,
  dimnames = list(c("Inter groupe", "Intra groupe", "Totale"),
                  c("SC", "dl", "CM", "F.obs", "Pr(F)"))
)
Anova.tableau

---
## Résumé

| Concept | Fonction R |
|---------|------------|
| Boxplot comparatif | `boxplot(x ~ facteur)` |
| ANOVA 1 facteur | `anova(lm())`, `aov()` |
| Contrastes | Test de Student sur combinaison linéaire |
| Normalité par groupe | `tapply(X, Groupe, shapiro.test)` |